[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/lstm_stock_prediction.ipynb)

# Prediction with LSTM


The current notebook performs timeseries prediction on NASDAQ index.

First, it is necessary to load the required libraries.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.manual_seed(1)

Set the processing device to accelerate the computation.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print('Using {} device'.format(device))

## Set the parameters

The overall parameters allow setting the previous `sequence_length` days and the train/test ratio.

In [ ]:
# set manual parameters
sequence_length = 20  # use previous 19 days to predict the 20th day
test_ratio = 0.2  # train on the first 80% data and test on the rest 20%

## Import dataset

 Import the NASDAQ index data and visualize it. The data must be sorted in chronological order (the file has the most recent days first).

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/xiptos/is_notes/refs/heads/main/datasets/ndx.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.astype({'Close/Last': 'float32', 'Open': 'float32', 'High': 'float32', 'Low': 'float32'})
df = df.sort_values(by='Date').reset_index(drop=True)  # oldest day first
df_view = df.set_index('Date')
print(df_view)

df_view['Close/Last'].interpolate().plot()
df_view['Open'].interpolate().plot()
df_view['High'].interpolate().plot()
df_view['Low'].interpolate().plot()
plt.legend(['Close', 'Open', 'High', 'Low'])
plt.show()
# we only predict on Close data in the following

In [ ]:
df.info()

## Create data loader

Each example is a window of `sequence_length` consecutive days: the first 19 days are the input and the last one is the target. The first 80% of the windows are used for training and the last 20% (the most recent days) for testing. Note that we can't shuffle the data before splitting it: we would be training with days from the future of the test set.

> Note: normalize the stock value to avoid node saturation.

The index grew a lot over the years: the test period has values higher than any value in the training period. So, instead of scaling all the values by the same constant, each window is divided by the value of its first day (minus 1). This way, the model learns relative changes (e.g., +2%), which are similar along the years.

In [ ]:
def load_data(stock_raw, sequence_length):
    data = []

    # create all possible sequences of length seq_len
    for index in range(len(stock_raw) - sequence_length + 1):
        data.append(stock_raw[index: index + sequence_length])

    data = np.array(data)
    test_set_size = int(np.round(test_ratio * data.shape[0]))
    train_set_size = data.shape[0] - (test_set_size)

    # normalize each window by the value of its first day
    base = data[:, :1]
    normalized = data / base - 1

    x_train = np.reshape(normalized[:train_set_size, :-1], (train_set_size, sequence_length - 1, 1))
    y_train = np.reshape(normalized[:train_set_size, -1], (train_set_size, 1))

    x_test = np.reshape(normalized[train_set_size:, :-1], (test_set_size, sequence_length - 1, 1))
    y_test = np.reshape(normalized[train_set_size:, -1], (test_set_size, 1))

    # values needed to convert the predictions back to the original scale
    base_test = base[train_set_size:]
    return [x_train, y_train, x_test, y_test, base_test]


x_train, y_train, x_test, y_test, base_test = load_data(df['Close/Last'].values, sequence_length)
print('x_train.shape = ', x_train.shape)
print('y_train.shape = ', y_train.shape)
print('x_test.shape = ', x_test.shape)
print('y_test.shape = ', y_test.shape)

In [ ]:
# make training and test sets in torch (float32: the MPS device doesn't support float64)
x_train = torch.from_numpy(x_train).float().to(device)
x_test = torch.from_numpy(x_test).float().to(device)
y_train = torch.from_numpy(y_train).float().to(device)
y_test = torch.from_numpy(y_test).float().to(device)

In [ ]:
n_steps = sequence_length - 1
batch_size = 20

train = torch.utils.data.TensorDataset(x_train, y_train)

train_loader = torch.utils.data.DataLoader(dataset=train, batch_size=batch_size, shuffle=True)

## Build the 2-layer stateless LSTM model

In [ ]:
# Build model
#####################
input_dim = 1
hidden_dim = 128
num_layers = 2
output_dim = 1


# Here we define our model as a class
class LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, device="cpu"):
        super().__init__()
        # Hidden dimensions
        self.hidden_dim = hidden_dim

        # Number of hidden layers
        self.num_layers = num_layers

        # Building your LSTM
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, device=device)

        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim, device=device)

    def forward(self, x):
        # stateless LSTM: the hidden and cell states start with zeros for every sequence
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Index hidden state of last time step
        out = self.fc(out[:, -1, :])
        return out

## Train

In [ ]:
# Train model
#####################
num_epochs = 100

hist = np.zeros(num_epochs)

model = LSTM(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers, device=device)

loss_fn = torch.nn.L1Loss()

optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
for t in range(num_epochs):

    for x_batch, y_batch in train_loader:
        # Forward pass
        y_train_pred = model(x_batch)

        loss = loss_fn(y_train_pred, y_batch)
        hist[t] += loss.item()

        # Zero out gradient, else they will accumulate between epochs
        optimiser.zero_grad()

        # Backward pass
        loss.backward()

        # Update parameters
        optimiser.step()

    hist[t] /= len(train_loader)  # average loss of the epoch
    if (t + 1) % 10 == 0:
        print("Epoch ", t + 1, "L1 loss (MAE): ", hist[t])

plt.plot(hist)
plt.xlabel("Epoch")
plt.ylabel("L1 loss")
plt.show()

## Save

Since training takes some time, lets keep the model on disk.

In [ ]:
PATH = "lstm_stock_prediction.pth"
torch.save(model.state_dict(), PATH)

## Load

Should we have already the model pretrained, let's load it.

In [ ]:
from pathlib import Path

# File path
file_path = Path("lstm_stock_prediction.pth")

# Check if the file exists
if file_path.is_file():
    model = LSTM(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers,
                 device=device)
    model.load_state_dict(torch.load(file_path, map_location=device, weights_only=True))
    model.eval()
else:
    print("File does not exist.")

## Visualize test result

Here we show the predictions over the real data in the test period, converted back to the original scale (index points).

To know if the model is any good, we compare it with a naive baseline: predicting that the value of tomorrow is the same as today. Predicting stock prices is very hard, and an LSTM often learns something close to this baseline.

> Exercise: try to beat the baseline, changing the `sequence_length`, the number of epochs, the size of the model or adding other features (e.g., `Open`, `High` and `Low`).

In [ ]:
# make predictions
model.eval()
with torch.no_grad():
    pred = model(x_test).cpu().numpy()

# convert back to the original scale
y_test_pred = (pred + 1) * base_test
y_test_true = (y_test.cpu().numpy() + 1) * base_test
# naive baseline: the value of the last day of the input window
y_test_naive = (x_test[:, -1, :].cpu().numpy() + 1) * base_test

rmse = lambda pred: math.sqrt(np.mean((pred - y_test_true) ** 2))
print('Test Score (LSTM):  %.2f RMSE' % rmse(y_test_pred))
print('Test Score (naive): %.2f RMSE' % rmse(y_test_naive))

# the date of each target is the last date of its window
test_dates = df['Date'].values[-len(y_test_true):]

plt.figure(figsize=(15, 8))
plt.plot(test_dates, y_test_true[:, 0])
plt.plot(test_dates, y_test_pred[:, 0])
plt.legend(['true stock value', 'predict value'])
plt.ylabel('Close stock value')
plt.show()